# This module demostrates , reading a csv file using spark and maimpulating dataframe in memory

In [1]:
from pyspark.sql import SparkSession
import os

# FORCE RESTART SESSION
try:
    spark.stop()
    print("Stopped existing session.")
except:
    pass

spark = (
    SparkSession.builder
    .appName("Orders ETL")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.cleanup-failures.ignored", "true")
    .config("spark.speculation", "false")
    .config("spark.sql.catalogImplementation", "hive")  # Ensure Hive support
    .enableHiveSupport()  # Enable Hive support explicitly
    .getOrCreate()
)

print("New Spark session created.")
print(spark.sparkContext.getConf().getAll())
# Verify parallelism
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")

New Spark session created.
[('spark.app.name', 'Orders ETL'), ('spark.app.startTime', '1779314353165'), ('spark.driver.port', '33591'), ('spark.executor.id', 'driver'), ('spark.app.submitTime', '1779314353002'), ('spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version', '2'), ('spark.hadoop.mapreduce.fileoutputcommitter.cleanup-failures.ignored', 'true'), ('spark.driver.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add

In [2]:
print(spark.sparkContext.getConf().getAll())

[('spark.app.name', 'Orders ETL'), ('spark.app.startTime', '1779314353165'), ('spark.driver.port', '33591'), ('spark.executor.id', 'driver'), ('spark.app.submitTime', '1779314353002'), ('spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version', '2'), ('spark.hadoop.mapreduce.fileoutputcommitter.cleanup-failures.ignored', 'true'), ('spark.driver.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.securi

## Reading the csv file using the spark read and au

In [3]:
if spark.sparkContext._jsc.sc().isStopped():
    raise RuntimeError("Spark session is stopped. Re-run Cell 1 to recreate the Spark session.")

orders = spark.read.csv(
    "/home/jovyan/work/data/raw/orders.csv",
    header=True,
    inferSchema=True
)
orders.show(5)

+--------+-----------+----------+--------+-----+-------------------+
|order_id|customer_id|product_id|quantity|price|    order_timestamp|
+--------+-----------+----------+--------+-----+-------------------+
|       1|        101|      P100|       2|  500|2026-01-01 10:00:00|
|       2|        102|      P101|       1|  200|2026-01-01 11:00:00|
|       3|        101|      P102|       3|  100|2026-01-02 09:00:00|
|       4|        103|      P103|       2|  700|2026-01-02 12:00:00|
+--------+-----------+----------+--------+-----+-------------------+



In [4]:
orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



# Manupulating data frame

Add new columns total_amount  =  price * quatity
                order date =  convert time stap to date

In [5]:
from pyspark.sql.functions import col, to_date

clean_orders = (
    orders
    .dropDuplicates()
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )
    .withColumn(
        "order_date",
        to_date(col("order_timestamp"))
    )
)

clean_orders.show()

+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|order_id|customer_id|product_id|quantity|price|    order_timestamp|total_amount|order_date|
+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|       3|        101|      P102|       3|  100|2026-01-02 09:00:00|         300|2026-01-02|
|       4|        103|      P103|       2|  700|2026-01-02 12:00:00|        1400|2026-01-02|
|       1|        101|      P100|       2|  500|2026-01-01 10:00:00|        1000|2026-01-01|
|       2|        102|      P101|       1|  200|2026-01-01 11:00:00|         200|2026-01-01|
+--------+-----------+----------+--------+-----+-------------------+------------+----------+



In [6]:
# Add validation to ensure data integrity
if orders.count() == 0:
    raise Exception("No data found in the raw orders file!")
if orders.filter(orders['order_id'].isNull()).count() > 0:
    raise Exception("Null values found in critical columns!")

## Writes the cleaned dataframe into parquet format

In [9]:
import os
import shutil

# This path is mapped to your Windows machine
volume_path = "/home/jovyan/work/data/processed/orders_parquet"

print(f"--- Saving to Volume: {volume_path} ---")

try:
    # 1. Clear existing data manually to avoid Spark 'overwrite' conflicts on Windows
    if os.path.exists(volume_path):
        print("Cleaning up old directory...")
        shutil.rmtree(volume_path)
    
    # 2. Collect to local and save using a Spark-compatible Parquet timestamp encoding
    print("Converting to local Pandas for reliable write...")
    pdf = clean_orders.toPandas()
    
    # Create directory
    os.makedirs(volume_path, exist_ok=True)
    
    # Force microsecond timestamp precision so Spark can read the file back
    target_file = os.path.join(volume_path, "orders.parquet")
    pdf.to_parquet(
        target_file,
        index=False,
        engine="pyarrow",
        coerce_timestamps="us",
        allow_truncated_timestamps=True
    )
    
    print(f"SUCCESS: File saved at {target_file}")
    
except Exception as e:
    print(f"ERROR: {str(e)}")

     
    print(f"SUCCESS: File saved at {target_file}")
    


--- Saving to Volume: /home/jovyan/work/data/processed/orders_parquet ---
Cleaning up old directory...
Converting to local Pandas for reliable write...
SUCCESS: File saved at /home/jovyan/work/data/processed/orders_parquet/orders.parquet


# Next Steps

The ETL pipeline has been successfully executed. Here are the next steps to ensure the pipeline's functionality:

1. **Verify Hive Table**:
   - Check the `processed_orders` table in Hive to ensure the data is loaded correctly.

2. **Airflow DAG**:
   - Re-trigger the Airflow DAG to validate the end-to-end pipeline.

3. **Data Validation**:
   - Perform data validation checks to ensure data integrity and correctness.

4. **Monitor Performance**:
   - Monitor the performance of the pipeline and optimize if necessary.

5. **Documentation**:
   - Document the pipeline process and any configurations for future reference.

# Verify Hive Table

Check the `processed_orders` table in Hive to ensure the data is loaded correctly.

In [12]:
##Verify Spark can read it back
print("\nVerifying Spark Read:")
verified_orders = spark.read.parquet(volume_path)
verified_orders.printSchema()
verified_orders.show(5, truncate=False)

# Create a Hive table for the processed data
hive_table = "processed_orders"
spark.sql(f"CREATE TABLE IF NOT EXISTS {hive_table} (order_id STRING, customer_id STRING, order_date DATE, total_amount FLOAT) STORED AS PARQUET")

# Load the Parquet data into the Hive table using the correct absolute path
clean_orders = spark.read.parquet(target_file)
clean_orders.write.mode("overwrite").saveAsTable(hive_table)

# Verify the data in the Hive table
spark.sql(f"SELECT * FROM {hive_table} LIMIT 10").show()



Verifying Spark Read:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_timestamp: timestamp_ntz (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|order_id|customer_id|product_id|quantity|price|order_timestamp    |total_amount|order_date|
+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|3       |101        |P102      |3       |100  |2026-01-02 09:00:00|300         |2026-01-02|
|4       |103        |P103      |2       |700  |2026-01-02 12:00:00|1400        |2026-01-02|
|1       |101        |P100      |2       |500  |2026-01-01 10:00:00|1000        |2026-01-01|
|2       |102        |P101      |1       |200  |2026-01-01 11:00:00|

# Re-trigger Airflow DAG

Re-run the Airflow DAG to validate the end-to-end pipeline automation.

In [13]:
# Re-trigger Airflow DAG
import os
os.system("airflow dags trigger orders_pipeline")

32512

# Perform Data Validation in Hive

Check for null values and verify the total number of rows in the `processed_orders` table.

In [11]:
# Check if Hive table exists
hive_table = "processed_orders"
existing_tables = spark.sql("SHOW TABLES").filter(f"tableName = '{hive_table}'")

if existing_tables.count() > 0:
    print(f"Table '{hive_table}' exists.")
    spark.sql(f"DESCRIBE FORMATTED {hive_table}").show(truncate=False)
else:
    print(f"Table '{hive_table}' does not exist.")

Table 'processed_orders' exists.
+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|order_id                    |string                                                        |NULL   |
|customer_id                 |string                                                        |NULL   |
|order_date                  |date                                                          |NULL   |
|total_amount                |float                                                         |NULL   |
|                            |                                                              |       |
|# Detailed Table Information|                                                              |       |
|Catalog                     |spark_catalog      